In [ ]:
import os
import glob
import pandas as pd
import numpy as np

# Paths
BASE_PATH = r"C:\Users\Srisha\Micro Imagine cup\NeuroAdaptive\data\hyperaktiv\raw"

CPT_PATH = os.path.join(BASE_PATH, "CPT_II_ConnersContinuousPerformanceTest.csv")
FEATURES_PATH = os.path.join(BASE_PATH, "features.csv")
PATIENT_INFO_PATH = os.path.join(BASE_PATH, "patient_info.csv")

ACTIVITY_PATH = os.path.join(BASE_PATH, "activity_data")
HRV_PATH = os.path.join(BASE_PATH, "hrv_data")

FULL_OUTPUT_PATH = os.path.join(BASE_PATH, "hyperaktiv_full_features.csv")
CORE_OUTPUT_PATH = os.path.join(BASE_PATH, "hyperaktiv_core_features.csv")


#Helper functions
def load_csv(path):
    return pd.read_csv(path, sep=";")

def extract_subject_id(filename):
    return int("".join(filter(str.isdigit, filename)))

def time_to_hour(series):
    return pd.to_datetime(series, errors="coerce").dt.hour

# Load metadata
print("Loading metadata...")
cpt = load_csv(CPT_PATH)
features = load_csv(FEATURES_PATH)
patient_info = load_csv(PATIENT_INFO_PATH)
cpt.rename(columns={cpt.columns[0]: "ID"}, inplace=True)
features.rename(columns={features.columns[0]: "ID"}, inplace=True)
patient_info.rename(columns={patient_info.columns[0]: "ID"}, inplace=True)

#HRV features
print("Processing HRV files...")
hrv_rows = []
for file in glob.glob(os.path.join(HRV_PATH, "*.csv")):
    sid = extract_subject_id(file)
    df = load_csv(file)
    df.columns = ["TIMESTAMP", "HRV"]
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"], errors="coerce")
    df["HRV"] = pd.to_numeric(df["HRV"], errors="coerce")
    df = df[(df["HRV"] >= 300) & (df["HRV"] <= 2000)]
    if df.empty:
        continue
    hrv_rows.append({
        "ID": sid,
        "hrv_mean": df["HRV"].mean(),
        "hrv_std": df["HRV"].std(),
        "hrv_min": df["HRV"].min(),
        "hrv_max": df["HRV"].max(),
        "hrv_count": len(df),
        "has_hrv": 1
    })
hrv_df = pd.DataFrame(hrv_rows)
# Features for activity data
print("Processing activity files...")
activity_rows = []
for file in glob.glob(os.path.join(ACTIVITY_PATH, "*.csv")):
    sid = extract_subject_id(file)
    df = load_csv(file)
    numeric_cols = df.select_dtypes(include=np.number).columns
    if len(numeric_cols) == 0:
        continue
    row = {"ID": sid}
    all_values = df[numeric_cols].values.flatten()
    row.update({
        "ACTIVITY_mean": np.nanmean(all_values),
        "ACTIVITY_std": np.nanstd(all_values),
        "ACTIVITY_min": np.nanmin(all_values),
        "ACTIVITY_max": np.nanmax(all_values),
        "has_activity": 1
    })
    activity_rows.append(row)
activity_df = pd.DataFrame(activity_rows)

#Merging the full dataset
print("Merging datasets...")
full_df = (
    patient_info
    .merge(features, on="ID", how="left")
    .merge(cpt, on="ID", how="left")
    .merge(activity_df, on="ID", how="left")
    .merge(hrv_df, on="ID", how="left")
)
# Availability flags
full_df["has_activity"] = full_df["has_activity"].fillna(0)
full_df["has_hrv"] = full_df["has_hrv"].fillna(0)

#Fixing time columns
for col in ["ACC_TIME", "HRV_TIME"]:
    if col in full_df.columns:
        full_df[f"{col.split('_')[0]}_hour"] = time_to_hour(full_df[col])
        full_df.drop(columns=col, inplace=True)
#Saving the full dataset
full_df.to_csv(FULL_OUTPUT_PATH, index=False)
print(f"Saved full feature dataset → {FULL_OUTPUT_PATH}")
#Making the core dataset for the model
core_columns = [
    "ID", "AGE", "SEX",
    # Label (keep ALL CPT columns)
    *[c for c in cpt.columns if c != "ID"],
    # Activity
    "ACTIVITY_mean", "ACTIVITY_std",
    "ACTIVITY_min", "ACTIVITY_max",
    "has_activity",
    # HRV
    "hrv_mean", "hrv_std",
    "hrv_min", "hrv_max",
    "hrv_count",
    "has_hrv",
]
core_df = full_df[[c for c in core_columns if c in full_df.columns]].copy()

#Handle missing values
numeric_cols = core_df.select_dtypes(include=np.number).columns
core_df[numeric_cols] = core_df[numeric_cols].fillna(0)

# Save dataset
core_df.to_csv(CORE_OUTPUT_PATH, index=False)

print("✅ Preprocessing COMPLETE")
print(f"Core dataset shape: {core_df.shape}")
print(f"Saved core dataset → {CORE_OUTPUT_PATH}")


Loading metadata...
Processing HRV files...
Processing activity files...
Merging datasets...


C:\Users\Srisha\AppData\Local\Temp\ipykernel_38960\2142416674.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce").dt.hour
C:\Users\Srisha\AppData\Local\Temp\ipykernel_38960\2142416674.py:31: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  return pd.to_datetime(series, errors="coerce").dt.hour


Saved full feature dataset → C:\Users\Srisha\Micro Imagine cup\NeuroAdaptive\data\hyperaktiv\raw\hyperaktiv_full_features.csv
✅ Preprocessing COMPLETE
Core dataset shape: (103, 799)
Saved core dataset → C:\Users\Srisha\Micro Imagine cup\NeuroAdaptive\data\hyperaktiv\raw\hyperaktiv_core_features.csv
